# Split Training Tiles into Batches for Manual Editing

This script finds all of the training tile images needed for each batch of the ARTS polygon updating process.

## Set-Up

In [64]:
import os
from google.cloud import storage
import google.auth
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
from pprint import pprint
import re
import itertools

In [ ]:
# Get GCS API Key
%load_ext dotenv
%dotenv

gcs_key = os.getenv("GCS_PL_ORDERS_KEY")

In [3]:
# ! gcloud auth login
gcloud_creds, _ = google.auth.default()

In [4]:
storage_client = storage.Client(project="AbruptThawMapping")
bucket_name = "abrupt_thaw"
bucket = storage_client.bucket(bucket_name)

## Import Data

In [ ]:
planet_grids = gpd.read_file(
    "../ARTS_metrics/data/planet_grids_2024_artsv.6.0.0_positive_polygons_stratified_random_order_by_region.geojson"  # or https://drive.google.com/file/d/1BdhvZrwUKd4vKJIehmpnuxqwlAXEwKT_/view?usp=drive_link
)
planet_grids

,year,id,grid_column,grid_row,basemap_name,delivery_location,link,original_order,region,n_tiles,expected_order_tiles,n_arts,expected_order_arts,stratified_random_order,chunk,geometry
0,2024,44-1514,44,1514,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/44/1514/,https://link.planet.com/basemaps/v1/mosaics/44...,155,1,117.0,5.0,759,8,1,1.0,"POLYGON ((-19156953.777 9588260.828, -19156953..."
1,2024,116-1560,116,1560,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/116/1...,https://link.planet.com/basemaps/v1/mosaics/44...,436,2,611.0,12.0,1420,11,2,1.0,"POLYGON ((-17748066.472 10488383.273, -1774806..."
2,2024,257-1533,257,1533,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/257/1...,https://link.planet.com/basemaps/v1/mosaics/44...,2003,3,2468.0,15.0,7190,18,3,1.0,"POLYGON ((-14988995.499 9960050.534, -14988995..."
3,2024,333-1613,333,1613,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/333/1...,https://link.planet.com/basemaps/v1/mosaics/44...,3400,4,3180.0,16.0,10878,19,4,1.0,"POLYGON ((-13501836.676 11525480.873, -1350183..."
4,2024,397-1513,397,1513,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/397/1...,https://link.planet.com/basemaps/v1/mosaics/44...,4613,5,81.0,4.0,210,3,5,1.0,"POLYGON ((-12249492.405 9568692.949, -12249492..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2989,2024,356-1647,356,1647,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/356/1...,https://link.planet.com/basemaps/v1/mosaics/44...,4065,4,3180.0,16.0,10878,19,2990,299.0,"POLYGON ((-13051775.454 12190788.767, -1305177..."
2990,2024,349-1643,349,1643,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/349/1...,https://link.planet.com/basemaps/v1/mosaics/44...,3893,4,3180.0,16.0,10878,19,2991,300.0,"POLYGON ((-13188750.608 12112517.25, -13188750..."
2991,2024,350-1640,350,1640,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/350/1...,https://link.planet.com/basemaps/v1/mosaics/44...,3918,4,3180.0,16.0,10878,19,2992,300.0,"POLYGON ((-13169182.729 12053813.612, -1316918..."
2992,2024,308-1567,308,1567,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/308/1...,https://link.planet.com/basemaps/v1/mosaics/44...,2908,4,3180.0,16.0,10878,19,2993,300.0,"POLYGON ((-13991033.657 10625358.428, -1399103..."


In [ ]:
training_tile_prefix = ("RTS_MODEL_V2/DATA/TEST/")
training_tile_directory = "gs://" + bucket_name + "/" + training_tile_prefix
blobs = bucket.list_blobs(prefix = training_tile_prefix)
files = [file for file in blobs if re.search(".tif", file.name)]
file_names = [file.name for file in files]

[<Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/10/1547/tile_10_1547_c5_r0.tif, 1763571824948030>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/10/1549/tile_10_1549_c2_r0.tif, 1763571828640210>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/10/1549/tile_10_1549_c3_r0.tif, 1763571828111622>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/10/1549/tile_10_1549_c4_r0.tif, 1763571828374077>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/10/1550/tile_10_1550_c2_r7.tif, 1763571829512102>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/10/1550/tile_10_1550_c3_r7.tif, 1763571829227434>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/100/1557/tile_100_1557_c2_r7.tif, 1763571836817976>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/100/1557/tile_100_1557_c3_r7.tif, 1763571836559765>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/101/1557/tile_101_1557_c4_r6.tif, 1763571846178428>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TEST/101/1557/tile_101_1557_c5_r6.tif, 1763571845919790>,
 <Blob: abrupt_thaw, RTS_MODEL_V2/DATA/TES

## Get Training Tiles for Each Batch

In [95]:
batches = planet_grids.chunk.astype(int).unique()

# training tile positions within Planet grid that are of interest - order matches output in adjacent_grids
tile_positions = [
    "c7_r0",
    "c7_r.",
    "c7_r7",
    "c._r7",
    "c0_r7",
    "c0_r.",
    "c0_r0",
    "c._r0"
]

# data frame to store output
training_tiles_df = pd.DataFrame(columns=["batch", "grid_id", "training_tile"])

for batch in batches:
    # print("--------------------------------------------")

    # get Planet grid IDs included in batch
    grid_ids = planet_grids[planet_grids.chunk == batch].id
    grid_ids = [id.replace("-", "_") for id in grid_ids]
    # print("All Grid IDs:", grid_ids)

    # Create empty list to keep training tile blobs in
    training_tiles = []
    
    # # Get all training tiles within the grid
    # patterns = "|".join(grid_ids)
    # contained_training_tiles = [file for file in files if re.search(patterns, file.name)]
    # # pprint(contained_training_tiles)

    # # Missing training tiles?
    # training_tile_grid_ids = list(pd.Series(
    #     [re.search("\\d{1,4}_\\d{1,4}", file.name).group(0) for file in contained_training_tiles]
    # ).unique())
    # print("Grid IDs in Training Tile File Names:", training_tile_grid_ids)
    # patterns = "|".join(training_tile_grid_ids)
    # missing_grids = [grid_id for grid_id in grid_ids if not re.search(patterns, grid_id)]
    # print("Missing IDs:", missing_grids)

    # Get all contained and adjacent training tiles
    for grid_id in grid_ids:

        # Get all contained training tiles (within the grid)
        contained_training_tiles = [file for file in files if re.search(grid_id, file.name)]
        # pprint(contained_training_tiles)
        
        # Get all adjacent training tiles
        # possible Planet columns in which adjacent training tiles may occur
        planet_column = int(grid_id.split("_")[0])
        planet_columns = list(range(planet_column - 1, planet_column + 2))
        # print(planet_columns)

        # possible Planet rows in which adjacent training tiles may occur
        planet_row = int(grid_id.split("_")[1])
        planet_rows = list(range(planet_row - 1, planet_row + 2))
        # print(planet_rows)

        # select only the training tiles immediately adjacent to the current Planet grid
        adjacent_grid_ids = ["_".join([str(element) for element in combo]) for combo in itertools.product(planet_columns, planet_rows)]
        adjacent_tile_ids = [
            grid + "_" + tile for grid, tile in zip(adjacent_grid_ids, tile_positions)
        ]
        # print(adjacent_tile_ids)

        # Get all training tiles that match
        patterns = "|".join(adjacent_tile_ids)
        # print(patterns)
        adjacent_training_tiles = [
            file
            for file in files
            if re.search(patterns, file.name)
        ]
        # pprint(adjacent_training_tiles)

        current_tiles = contained_training_tiles + adjacent_training_tiles
        training_tiles = training_tiles + current_tiles

        training_tiles_df = pd.concat(
            [
                training_tiles_df,
                pd.DataFrame(
                    {
                        "batch": batch,
                        "grid_id": grid_id,
                        "training_tile": [current_tile.name for current_tile in current_tiles],
                    }
                ),
            ]
        )

   # training_tiles can be used to read in all of the tifs and merge them into one file

# training_tiles_df contains all the relevant imagery training_tiles for the current Planet grid
training_tiles_df.to_csv("./data/training_tiles_by_batch.csv", index = False)
training_tiles_df

,batch,grid_id,training_tile
0,1,44_1514,RTS_MODEL_V2/DATA/TEST/44/1514/tile_44_1514_c0...
1,1,44_1514,RTS_MODEL_V2/DATA/TEST/44/1514/tile_44_1514_c0...
2,1,44_1514,RTS_MODEL_V2/DATA/TEST/44/1514/tile_44_1514_c1...
3,1,44_1514,RTS_MODEL_V2/DATA/TEST/43/1514/tile_43_1514_c7...
4,1,44_1514,RTS_MODEL_V2/DATA/TEST/43/1514/tile_43_1514_c7...
...,...,...,...
11,300,350_1640,RTS_MODEL_V2/DATA/TEST/349/1640/tile_349_1640_...
12,300,350_1640,RTS_MODEL_V2/DATA/TEST/349/1640/tile_349_1640_...
13,300,350_1640,RTS_MODEL_V2/DATA/TEST/349/1640/tile_349_1640_...
0,300,308_1567,RTS_MODEL_V2/DATA/TEST/308/1567/tile_308_1567_...
